# SEP_CFE_DiCE Quickstart

This notebook demonstrates a complete constrained-counterfactual workflow using the standalone `sep-cfe-dice` package.

## 1) Install package (editable)

Run this once if you opened the notebook from `examples/`.

In [2]:
%pip install -e ".."

Defaulting to user installation because normal site-packages is not writeable
Obtaining file:///Users/pranjal/PycharmProjects/PGCE
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sep-cfe-dice (pyproject.toml) ... done
  Created wheel for sep-cfe-dice: filename=sep_cfe_dice-0.1.0-0.editable-py3-none-any.whl size=3605 sha256=0933cd6df0d007bc4519a3c0142876fa4ec3e173ff9860e84110d411df1860c2
  Stored in directory: /private/var/folders/f7/zbff57wd2h14dcx8bnpxwkk80000gn/T/pip-ephem-wheel-cache-e5scqj_v/wheels/26/be/f4/db9c9932454eb1bbfdb5565507dc03e6e3254f0e378411831f
Successfully built sep-cfe-dice

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use up

## 2) Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

from SEP_CFE_DiCE import (
    FeatureRangeConstraint,
    OrderedFeaturesConstraint,
    FeatureThresholdConstraint,
    build_constrained_explainer,
    generate_counterfactuals,
    extract_first_counterfactual_df,
    plot_counterfactual_deltas,
    plot_counterfactual_profiles,
)

## 3) Create synthetic training/query data

In [2]:
X, y = make_classification(
    n_samples=600,
    n_features=6,
    n_informative=5,
    n_redundant=0,
    random_state=42,
)

feature_names = [f"f{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y.astype(int)

train_df = df.iloc[:500].reset_index(drop=True)
query_df = df.iloc[500:501].drop(columns=["target"]).reset_index(drop=True)

train_df.head()

,f0,f1,f2,f3,f4,f5,target
0,-1.020147,0.559700,2.094974,-1.341032,-2.204612,0.638505,1
1,1.641411,-0.239507,0.130662,0.430385,-1.797441,0.013403,0
2,1.878070,-1.270409,1.571174,0.475997,0.394085,-1.153230,1
3,1.123992,-1.796738,-0.863854,1.274422,2.565763,-1.017824,1
4,-3.343247,1.182566,4.392642,-3.107437,-1.897342,-0.049529,1


## 4) Train a model

In [3]:
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(train_df.drop(columns=["target"]), train_df["target"])

RandomForestClassifier(n_estimators=200, random_state=42)

## 5) Define constraints

In [4]:
permitted_ranges = {
    col: [float(train_df[col].quantile(0.01)), float(train_df[col].quantile(0.99))]
    for col in feature_names
}

constraints = [
    FeatureRangeConstraint(ranges=permitted_ranges, penalty_value=500.0),
    OrderedFeaturesConstraint(
        ordered_feature_groups=[("f0", "f1", "f2")],
        increasing=False,
        strict=False,
        penalty_value=50.0,
    ),
    FeatureThresholdConstraint(
        feature_name="f0",
        upper_bound=float(train_df["f0"].quantile(0.95)),
        penalty_value=100.0,
    ),
]

constraints

[FeatureRangeConstraint(ranges={'f0': [-3.4660815403048906, 3.1746663811446716], 'f1': [-2.3703142838285327, 2.278565652579413], 'f2': [-2.463864276976251, 3.8086258381043954], 'f3': [-4.153263995474041, 3.839124791550333], 'f4': [-3.8747694168212328, 3.3430124450991374], 'f5': [-3.4812718113830505, 3.6670130771465734]}, penalty_value=500.0, strict_bounds=False),
 OrderedFeaturesConstraint(ordered_feature_groups=[('f0', 'f1', 'f2')], penalty_value=50.0, increasing=False, strict=False, skip_missing=True),
 FeatureThresholdConstraint(feature_name='f0', lower_bound=None, upper_bound=2.4557955084702954, lower_inclusive=True, upper_inclusive=True, penalty_value=100.0)]

## 6) Build constrained explainer

In [5]:
explainer, interfaces = build_constrained_explainer(
    dataframe=train_df,
    model=model,
    outcome_name="target",
    constraints=constraints,
    l0_penalty_weight=0.2,
)

## 7) Generate counterfactuals

Depending on your machine this can take a little while.

In [6]:
cf_obj = generate_counterfactuals(
    explainer=explainer,
    query_instances=query_df,
    total_cfs=3,
    desired_class=1,
    maxiterations=300,
    population_size=40,
    proximity_weight=0.2,
    sparsity_weight=0.2,
    diversity_weight=2.0,
)

cf_df = extract_first_counterfactual_df(cf_obj)
cf_df

  0%|          | 0/1 [00:00<?, ?it/s]


TypeError: _generate_counterfactuals() got an unexpected keyword argument 'population_size'

## 8) Visualize feature deltas and profiles

In [7]:
feature_cf_df = cf_df.drop(columns=["target"], errors="ignore")
_ = plot_counterfactual_deltas(query_df.iloc[[0]], feature_cf_df, top_k=6)
_ = plot_counterfactual_profiles(query_df.iloc[[0]], feature_cf_df, top_k=6, max_counterfactuals=3)

NameError: name 'cf_df' is not defined